# UCF101 Corrected Motion-Aware BiLSTM + Attention Experiment

**Corrected version:** the pose-sequence builder stores each frame as 17 consecutive `(x, y, confidence)` triples. This notebook reshapes to `[17, 3]` first and selects x/y from every joint before calculating velocity and acceleration.

- Original pose: 51 features
- Joint velocity: 34 features
- Joint acceleration: 34 features
- Total: 119 features per frame

It trains a fresh model from scratch and saves to separate `motion119_corrected` checkpoint/report filenames. It does not overwrite your existing best checkpoint. Feature normalization is fit on training data only; validation macro-F1 selects the checkpoint; test metrics are computed after training.

Run cells top to bottom. Confirm `PROJECT_DIR` is correct before starting.

In [1]:
from pathlib import Path
import json, random, time
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

PROJECT_DIR = Path(r"C:\Users\admin\ViTPose_Action_Recognition")
DATA_DIR = PROJECT_DIR / "data" / "processed" / "sequences_all"
MODEL_DIR = PROJECT_DIR / "outputs" / "models"
REPORT_DIR = PROJECT_DIR / "outputs" / "reports"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

SEED, BATCH_SIZE, EPOCHS, PATIENCE = 42, 64, 50, 8
LR, WEIGHT_DECAY, HIDDEN_SIZE, DROPOUT = 3e-4, 1e-4, 96, 0.35
LABEL_SMOOTHING, NUM_WORKERS = 0.05, 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print("Device:", DEVICE, "| data:", DATA_DIR)
assert DATA_DIR.is_dir(), f"Dataset folder not found: {DATA_DIR}"


Device: cuda | data: C:\Users\admin\ViTPose_Action_Recognition\data\processed\sequences_all


In [2]:
def load(name):
    p = DATA_DIR / name
    assert p.is_file(), f"Missing {p}"
    return np.load(p, mmap_mode="r", allow_pickle=False)

Xtr0, ytr = load("X_train.npy"), np.asarray(load("y_train.npy"), dtype=np.int64)
Xva0, yva = load("X_val.npy"), np.asarray(load("y_val.npy"), dtype=np.int64)
Xte0, yte = load("X_test.npy"), np.asarray(load("y_test.npy"), dtype=np.int64)
vtr, vva, vte = load("videos_train.npy"), load("videos_val.npy"), load("videos_test.npy")
assert Xtr0.shape[2] == 51 and Xtr0.shape[1:] == Xva0.shape[1:] == Xte0.shape[1:]
assert len(Xtr0)==len(ytr)==len(vtr) and len(Xva0)==len(yva)==len(vva) and len(Xte0)==len(yte)==len(vte)
N_CLASSES = int(max(ytr.max(), yva.max(), yte.max()) + 1)
print("Train/val/test:", Xtr0.shape, Xva0.shape, Xte0.shape, "| classes:", N_CLASSES)


Train/val/test: (63533, 16, 51) (20980, 16, 51) (21009, 16, 51) | classes: 101


## Create the 119-feature representation

Velocity and acceleration are computed along each window's time axis. The first time step is zero-padded. Arrays are processed in chunks during feature construction, then retained in memory for training.

In [3]:
def add_motion(X, chunk_size=1024):
    """
    Input X: [N, T, 51], with each joint stored as (x, y, confidence).
    Output: [N, T, 119] = original pose (51) + velocity (34) + acceleration (34).
    """
    n, t, f = X.shape
    assert f == 51, f"Expected 51 features, got {f}"
    out = np.empty((n, t, 119), dtype=np.float32)

    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        pose = np.asarray(X[start:end], dtype=np.float32)

        # IMPORTANT: reshape triples first, then select x/y for every joint.
        # Do NOT use pose[:, :, :34], because confidence values are interleaved.
        joints = pose.reshape(end - start, t, 17, 3)
        xy = joints[:, :, :, :2]  # [batch, time, 17 joints, (x,y)]

        velocity = np.zeros_like(xy)
        acceleration = np.zeros_like(xy)
        if t > 1:
            velocity[:, 1:] = xy[:, 1:] - xy[:, :-1]
            acceleration[:, 1:] = velocity[:, 1:] - velocity[:, :-1]

        out[start:end, :, :51] = pose
        out[start:end, :, 51:85] = velocity.reshape(end - start, t, 34)
        out[start:end, :, 85:119] = acceleration.reshape(end - start, t, 34)

    return out


# Quick sanity check on a small sample before processing the full dataset.
sample = np.asarray(Xtr0[:2], dtype=np.float32)
sample_motion = add_motion(sample)
assert sample_motion.shape == (len(sample), sample.shape[1], 119)
# At t=0, velocity and acceleration are initialized to zero.
assert np.allclose(sample_motion[:, 0, 51:], 0.0)
print("Corrected feature indexing verified:", sample_motion.shape)

print("Building motion features...")
Xtr, Xva, Xte = add_motion(Xtr0), add_motion(Xva0), add_motion(Xte0)
print(Xtr.shape, Xva.shape, Xte.shape)

Corrected feature indexing verified: (2, 16, 119)
Building motion features...
(63533, 16, 119) (20980, 16, 119) (21009, 16, 119)


In [4]:
# Fit normalization statistics on TRAINING data only.
mu = Xtr.reshape(-1, 119).mean(axis=0, dtype=np.float64).astype(np.float32)
sd = Xtr.reshape(-1, 119).std(axis=0, dtype=np.float64).astype(np.float32)
sd[sd < 1e-6] = 1.0
def normalize(X):
    return ((X - mu[None, None, :]) / sd[None, None, :]).astype(np.float32)
Xtr, Xva, Xte = normalize(Xtr), normalize(Xva), normalize(Xte)
print("Normalized features. Train shape:", Xtr.shape)


Normalized features. Train shape: (63533, 16, 119)


In [5]:
class PoseDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X, self.y, self.augment = X, y, augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        x = self.X[i].copy()
        if self.augment:
            # Small jitter on normalized pose/motion features only.
            # Keep the original confidence channels (2, 5, ..., 50) unchanged.
            noise = np.random.normal(0, 0.01, x.shape).astype(np.float32)
            confidence_idx = np.arange(2, 51, 3)
            noise[:, confidence_idx] = 0.0
            x += noise
        return torch.from_numpy(x), torch.tensor(int(self.y[i]), dtype=torch.long)

train_loader = DataLoader(PoseDataset(Xtr, ytr, True), batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=DEVICE.type=="cuda")
val_loader = DataLoader(PoseDataset(Xva, yva), batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=DEVICE.type=="cuda")
test_loader = DataLoader(PoseDataset(Xte, yte), batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=DEVICE.type=="cuda")
print("Batches train/val/test:", len(train_loader), len(val_loader), len(test_loader))

Batches train/val/test: 993 328 329


In [6]:
class MotionBiLSTMAttention(nn.Module):
    def __init__(self, input_size=119, hidden=96, classes=101, dropout=0.35):
        super().__init__()
        self.norm = nn.LayerNorm(input_size)
        self.lstm = nn.LSTM(input_size, hidden, batch_first=True, bidirectional=True)
        self.attn = nn.Sequential(nn.Linear(hidden*2, hidden), nn.Tanh(), nn.Linear(hidden, 1))
        self.head = nn.Sequential(nn.LayerNorm(hidden*2), nn.Dropout(dropout), nn.Linear(hidden*2, classes))
    def forward(self, x):
        seq, _ = self.lstm(self.norm(x))
        weights = torch.softmax(self.attn(seq).squeeze(-1), dim=1)
        context = torch.sum(seq * weights.unsqueeze(-1), dim=1)
        return self.head(context)

model = MotionBiLSTMAttention(classes=N_CLASSES, hidden=HIDDEN_SIZE, dropout=DROPOUT).to(DEVICE)
loss_fn = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

def run_epoch(loader, training):
    model.train(training)
    loss_sum, ys, ps = 0.0, [], []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        if training: optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = model(xb); loss = loss_fn(logits, yb)
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        loss_sum += loss.item() * len(yb)
        ys.append(yb.detach().cpu().numpy()); ps.append(logits.argmax(1).detach().cpu().numpy())
    yt, yp = np.concatenate(ys), np.concatenate(ps)
    return {"loss": loss_sum/len(loader.dataset), "accuracy": accuracy_score(yt, yp),
            "macro_f1": f1_score(yt, yp, labels=np.arange(N_CLASSES), average="macro", zero_division=0)}


## Train and save the best validation checkpoint

Outputs use distinct `motion119` filenames so your previous model remains untouched.

In [7]:
BEST_PATH = MODEL_DIR / "lstm_bilstm_attention_motion119_corrected_best.pt"
LAST_PATH = MODEL_DIR / "lstm_bilstm_attention_motion119_corrected_last.pt"
HISTORY_PATH = REPORT_DIR / "motion119_corrected_training_history.csv"
history, best_f1, best_epoch, stale = [], -1.0, 0, 0

for epoch in range(1, EPOCHS+1):
    start = time.time()
    tr = run_epoch(train_loader, True)
    va = run_epoch(val_loader, False)
    scheduler.step(va["macro_f1"])
    row = {"epoch":epoch, "train_loss":tr["loss"], "train_accuracy":tr["accuracy"],
           "train_macro_f1":tr["macro_f1"], "val_loss":va["loss"],
           "val_accuracy":va["accuracy"], "val_macro_f1":va["macro_f1"],
           "lr":optimizer.param_groups[0]["lr"], "seconds":time.time()-start}
    history.append(row); pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)
    if va["macro_f1"] > best_f1:
        best_f1, best_epoch, stale = va["macro_f1"], epoch, 0
        torch.save({"model_state":model.state_dict(), "input_size":119, "hidden_size":HIDDEN_SIZE,
                    "num_classes":N_CLASSES, "dropout":DROPOUT, "best_epoch":best_epoch,
                    "best_val_macro_f1":best_f1, "feature_mean":mu, "feature_std":sd, "seed":SEED}, BEST_PATH)
    else:
        stale += 1
    torch.save({"model_state":model.state_dict(), "epoch":epoch, "optimizer_state":optimizer.state_dict(),
                "best_epoch":best_epoch, "best_val_macro_f1":best_f1}, LAST_PATH)
    print(f"Epoch {epoch:02d}/{EPOCHS} | train loss={tr['loss']:.4f} acc={tr['accuracy']:.4f} F1={tr['macro_f1']:.4f} | val loss={va['loss']:.4f} acc={va['accuracy']:.4f} F1={va['macro_f1']:.4f} | best val F1={best_f1:.4f} epoch={best_epoch}")
    if stale >= PATIENCE:
        print("Early stopping."); break

print("Best checkpoint:", BEST_PATH)
print("History:", HISTORY_PATH)


Epoch 01/50 | train loss=2.7882 acc=0.3840 F1=0.2475 | val loss=2.2215 acc=0.5136 F1=0.3528 | best val F1=0.3528 epoch=1
Epoch 02/50 | train loss=1.9398 acc=0.5888 F1=0.4541 | val loss=1.9190 acc=0.5933 F1=0.4692 | best val F1=0.4692 epoch=2
Epoch 03/50 | train loss=1.6367 acc=0.6712 F1=0.5575 | val loss=1.7826 acc=0.6269 F1=0.5137 | best val F1=0.5137 epoch=3
Epoch 04/50 | train loss=1.4521 acc=0.7248 F1=0.6257 | val loss=1.7115 acc=0.6496 F1=0.5420 | best val F1=0.5420 epoch=4
Epoch 05/50 | train loss=1.3287 acc=0.7604 F1=0.6716 | val loss=1.6917 acc=0.6579 F1=0.5545 | best val F1=0.5545 epoch=5
Epoch 06/50 | train loss=1.2321 acc=0.7887 F1=0.7085 | val loss=1.6672 acc=0.6660 F1=0.5657 | best val F1=0.5657 epoch=6
Epoch 07/50 | train loss=1.1555 acc=0.8142 F1=0.7427 | val loss=1.6643 acc=0.6694 F1=0.5706 | best val F1=0.5706 epoch=7
Epoch 08/50 | train loss=1.0941 acc=0.8314 F1=0.7704 | val loss=1.6665 acc=0.6674 F1=0.5676 | best val F1=0.5706 epoch=7
Epoch 09/50 | train loss=1.0390 

## Final test evaluation

Run this only after training finishes. It reports window-level metrics and video-level metrics by averaging each video's window probabilities.

In [8]:
ckpt = torch.load(BEST_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state"]); model.eval()
prob_list, true_list = [], []
with torch.inference_mode():
    for xb, yb in test_loader:
        probs = torch.softmax(model(xb.to(DEVICE)), dim=1).cpu().numpy()
        prob_list.append(probs); true_list.append(yb.numpy())
probs = np.concatenate(prob_list); true = np.concatenate(true_list).astype(np.int64)
pred = probs.argmax(1)

video_ids = np.asarray(vte)
unique_videos, inverse = np.unique(video_ids, return_inverse=True)
sums = np.zeros((len(unique_videos), N_CLASSES), dtype=np.float64)
np.add.at(sums, inverse, probs)
counts = np.bincount(inverse, minlength=len(unique_videos))
video_probs = sums / counts[:, None]
video_true = np.empty(len(unique_videos), dtype=np.int64)
for i in range(len(unique_videos)):
    labels = np.unique(true[inverse == i])
    if len(labels) != 1: raise ValueError(f"Inconsistent labels for video {unique_videos[i]!r}: {labels}")
    video_true[i] = labels[0]
video_pred = video_probs.argmax(1)

results = {
    "checkpoint": str(BEST_PATH), "best_epoch": int(ckpt["best_epoch"]),
    "best_validation_macro_f1": float(ckpt["best_val_macro_f1"]),
    "test_windows": int(len(true)), "test_videos": int(len(unique_videos)),
    "window_level_accuracy": float(accuracy_score(true, pred)),
    "window_level_balanced_accuracy": float(balanced_accuracy_score(true, pred)),
    "window_level_macro_f1": float(f1_score(true, pred, labels=np.arange(N_CLASSES), average="macro", zero_division=0)),
    "video_level_accuracy": float(accuracy_score(video_true, video_pred)),
    "video_level_balanced_accuracy": float(balanced_accuracy_score(video_true, video_pred)),
    "video_level_macro_f1": float(f1_score(video_true, video_pred, labels=np.arange(N_CLASSES), average="macro", zero_division=0))
}
with open(REPORT_DIR / "motion119_corrected_test_metrics.json", "w", encoding="utf-8") as f: json.dump(results, f, indent=2)
pd.DataFrame({"video_id":unique_videos.astype(str), "true_label":video_true, "predicted_label":video_pred,
              "number_of_windows":counts, "correct":video_true==video_pred}).to_csv(REPORT_DIR / "motion119_corrected_video_predictions.csv", index=False)
print(json.dumps(results, indent=2))
print("Saved reports to:", REPORT_DIR)


{
  "checkpoint": "C:\\Users\\admin\\ViTPose_Action_Recognition\\outputs\\models\\lstm_bilstm_attention_motion119_corrected_best.pt",
  "best_epoch": 18,
  "best_validation_macro_f1": 0.5834719029297952,
  "test_windows": 21009,
  "test_videos": 2441,
  "window_level_accuracy": 0.6532438478747203,
  "window_level_balanced_accuracy": 0.5632841415704871,
  "window_level_macro_f1": 0.5644938642586417,
  "video_level_accuracy": 0.6665301106104056,
  "video_level_balanced_accuracy": 0.6351474086538804,
  "video_level_macro_f1": 0.6268357163753548
}
Saved reports to: C:\Users\admin\ViTPose_Action_Recognition\outputs\reports


## Notes and checks

- This notebook corrects the indexing bug in the previous motion notebook: `pose.reshape(N, T, 17, 3)[..., :2]` extracts x/y from each joint while excluding interleaved confidence values.
- The sequence-generation notebook confirms that the stored 51 features are flattened from `[17, 3]`, so this indexing matches the actual data format.
- If RAM is limited, reduce `BATCH_SIZE` to 16 or 32. If feature arrays themselves exhaust RAM, stop and ask for a disk-backed version.
- CUDA is used automatically when available; CPU training can be slow.
- New files use `motion119_corrected` names. The original `lstm_bilstm_attention_best.pt` is not overwritten.
- Compare video-level accuracy to your previous video-level baseline (about 70.18%), and macro-F1 to macro-F1. These metrics may still be lower; 75% is a goal, not a guarantee.